# Product Experiment Walkthrough

This notebook is a reviewer-oriented walkthrough of the tested experiment pipeline. It reuses repository modules for assignment health, the primary effect, power/MDE, device interaction, and CUPED-style sensitivity analysis instead of reimplementing formulas in notebook-only code.

The data are deterministic and synthetic. The unadjusted 7-day activation analysis remains the confirmatory result.

In [ ]:
from src.generate_dataset import generate_users
from src.experiment import two_proportion_test
from src.diagnostics import sample_ratio_mismatch, randomization_balance, treatment_interaction
from src.power import minimum_detectable_effect, sample_size_per_arm, two_sided_power
from src.cuped import cuped_adjust_activation


In [ ]:
rows = generate_users(12_000, seed=20_260_808)
primary = two_proportion_test(rows, 'activated_7d')
n_control = sum(row['variant'] == 'control' for row in rows)
n_treatment = sum(row['variant'] == 'treatment' for row in rows)
print(f'control: {n_control:,} users @ {primary.control_rate:.2%}')
print(f'treatment: {n_treatment:,} users @ {primary.treatment_rate:.2%}')
print(f'activation lift: {primary.absolute_lift * 100:+.2f} pp')
print(f'p-value: {primary.p_value:.4f}; 95% CI {primary.ci_low * 100:+.2f} to {primary.ci_high * 100:+.2f} pp')


In [ ]:
srm = sample_ratio_mismatch(rows)
balance = randomization_balance(rows)
interaction = treatment_interaction(rows, segment='device', segment_a='desktop', segment_b='mobile', metric='activated_7d')
print(f'SRM p-value: {srm.p_value:.4f}')
print(f'max pre-treatment |SMD|: {max(abs(item.standardized_difference) for item in balance):.3f}')
print(f'desktop-minus-mobile lift interaction: {interaction.interaction_effect * 100:+.2f} pp; p={interaction.p_value:.4f}')


In [ ]:
realized_mde = minimum_detectable_effect(primary.control_rate, n_control, n_treatment, target_power=0.80)
observed_power = two_sided_power(primary.control_rate, primary.absolute_lift, n_control, n_treatment)
sample_for_2pp = sample_size_per_arm(primary.control_rate, 0.02, target_power=0.80)
print(f'80% power MDE: {realized_mde * 100:.2f} pp')
print(f'planning power at observed +{primary.absolute_lift * 100:.2f} pp: {observed_power:.1%}')
print(f'balanced users / arm for +2.00 pp at 80% power: {sample_for_2pp:,}')


In [ ]:
cuped = cuped_adjust_activation(rows)
print(f'CUPED-style variance reduction: {cuped.variance_reduction:.2%}')
print(f'raw lift: {cuped.raw_difference * 100:+.2f} pp')
print(f'adjusted lift: {cuped.adjusted_difference * 100:+.2f} pp')
print(f'adjusted p-value: {cuped.p_value:.4f}')


## Interpretation discipline

- Check assignment integrity before interpreting outcomes.
- MDE and power answer design questions; they are not post-hoc significance thresholds.
- The device interaction is suggestive but not confirmed at alpha = 0.05.
- The CUPED-style result is a sensitivity/precision demonstration because this new-user experiment has no genuine pre-period activation outcome.
- Product decisions remain anchored to the pre-specified unadjusted primary analysis and guardrails.